Project 3, Person A: Task 3.1, Task 3.2, and the Part II AI exercise

Code follows the official documentation examples of the tools we use. Every code cell starts with a comment that says either Source, naming the documentation page and example the code follows, or Custom, saying why no example fits and what reference the code is based on. The companion document, Project 3 Person A Reference, links each decision to its documentation example.

To run it, data/Online_Retail.xlsx and data/groceries.csv must sit in a data folder beside this notebook.

In [ ]:
%pip install mlxtend openpyxl

In [ ]:
import csv
import tracemalloc

import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

Task 3.1: Data Preprocessing

In [ ]:
# loading retail data 
retail = pd.read_excel("data/Online_Retail.xlsx", dtype={"InvoiceNo": str, "StockCode": str})
print(retail.shape)

In [ ]:
# remove missing item ID's 
retail = retail.dropna(subset=["StockCode", "Description","CustomerID"])                  

# cleaning other items 
    # cancellations and adjustments 
    # returns 
    # stock notes 
    # non product codes 
retail = retail[~retail["InvoiceNo"].str.startswith(("C", "c", "A"))          
    & (retail["Quantity"] > 0) 
    & (retail["UnitPrice"] > 0)         
    & retail["StockCode"].str.fullmatch(r"\d{5}[A-Za-z]*")]       

# combines stock codes with mixed cases 
retail["StockCode"] = retail["StockCode"].str.upper()                          
print(retail.shape, "| invoices:", retail["InvoiceNo"].nunique(), "| products:", retail["StockCode"].nunique())

In [ ]:
# panda user guide for aggregation 
retail_baskets = retail.groupby("InvoiceNo")["StockCode"].agg(lambda x: set(x)).tolist()

# loading grocercy data 
with open("data/groceries.csv", newline="") as csvfile:
    groceries_baskets = list(csv.reader(csvfile))

print("Online Retail baskets:", len(retail_baskets), "| Groceries baskets:", len(groceries_baskets))

In [ ]:
# mlxtend datafram examples  
te = TransactionEncoder()
te_ary = te.fit(retail_baskets).transform(retail_baskets)
retail_df = pd.DataFrame(te_ary, columns=te.columns_)

te = TransactionEncoder()
te_ary = te.fit(groceries_baskets).transform(groceries_baskets)
groceries_df = pd.DataFrame(te_ary, columns=te.columns_)

print("Online Retail:", retail_df.shape, "| Groceries:", groceries_df.shape)

Task 3.2: Apriori vs. FP-Growth

In [ ]:
# Source: mlxtend fpgrowth user guide, Example 2 (Apriori versus FPGrowth): these are its three calls
SETTINGS = {
    "apriori": lambda df, ms: apriori(df, min_support=ms),
    "apriori low_memory": lambda df, ms: apriori(df, min_support=ms, low_memory=True),
    "fpgrowth": lambda df, ms: fpgrowth(df, min_support=ms),
}
DATASETS = {"Groceries": groceries_df, "Online Retail": retail_df}

# specs range
MINSUP_GRID = [0.05, 0.02, 0.01, 0.005, 0.002, 0.001]

# needed do to failure on running out of memory on laptop
NOT_RUN = {("Online Retail", "apriori", ms) for ms in [0.01, 0.005, 0.002, 0.001]}

In [ ]:
results = []
for dataset, df in DATASETS.items():
    for setting, run in SETTINGS.items():
        for ms in MINSUP_GRID:
            if (dataset, setting, ms) in NOT_RUN:
                results.append({"dataset": dataset, "setting": setting, "min_support": ms, "status": "not run (memory)"})
                continue

            # time runs 
            timing = %timeit -o -q -n 1 -r 3 run(df, ms)

            # spec req for peak mem usage 
            tracemalloc.start()
            run(df, ms)
            current, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            results.append({"dataset": dataset, "setting": setting, "min_support": ms, "status": "ok",
                            "runtime_s": timing.best, "peak_mb": peak / 1e6})
            print(dataset, "|", setting, "|", ms, "| done")

pd.DataFrame(results)

Task 3.3: Rule
Generation & Filtering

In [ ]:
import numpy as np
from mlxtend.frequent_patterns import association_rules

#setting everything up for tast 3.3
DATASET_NAME = "Groceries"  
ALGORITHM = "fpgrowth"      
MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.40
MIN_LIFT = 1.0
H_THRESHOLD = 0.60
CROSS_SUPPORT_THRESHOLD = 0.30

basket_df = DATASETS[DATASET_NAME]

#setting up the thresholds and settings for confidence and Lift
frequent_itemsets = SETTINGS[ALGORITHM](basket_df, MIN_SUPPORT)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

print(f"{DATASET_NAME}: {len(basket_df):,} transactions, {len(frequent_itemsets):,} frequent itemsets")
display(frequent_itemsets.head())

In [ ]:
import inspect

#this function generates rules by taking the itemset , the transactions and the miin confidence and min lift
def generate_rules(itemsets, num_transactions, min_confidence=0.4, min_lift=1.0):
    if itemsets.empty or not any(len(x) >= 2 for x in itemsets['itemsets']):
        return pd.DataFrame()
    
    kwargs = dict(df=itemsets, metric='confidence', min_threshold=min_confidence)
    
    if 'num_itemsets' in inspect.signature(association_rules).parameters:
        kwargs['num_itemsets'] = num_transactions
        
    rules = association_rules(**kwargs)
    rules = rules.loc[rules['lift'] >= min_lift].copy()
    
    return rules.sort_values(['lift', 'confidence'], ascending=False).reset_index(drop=True)

rules = generate_rules(frequent_itemsets, len(basket_df), MIN_CONFIDENCE, MIN_LIFT)

print('Filtered rules:', len(rules))

display(rules[['antecedents','consequents','support','confidence','lift']].head(15) if not rules.empty else rules)

In [ ]:
def remove_redundant_rules(rules, tol=1e-10):
    if rules.empty:
        return rules.copy()
    
    keep = []
    
    for i, r in rules.iterrows():
        redundant = False
        
        for j, q in rules.iterrows():
            if i == j or r['consequents'] != q['consequents']:
                continue
        
            if q['antecedents'] < r['antecedents'] and \
               np.isclose(q['support'], r['support'], atol=tol, rtol=0) and \
               np.isclose(q['confidence'], r['confidence'], atol=tol, rtol=0):
                    redundant = True
                    break
        keep.append(not redundant)
        
    return rules.loc[keep].reset_index(drop=True)

nonredundant_rules = remove_redundant_rules(rules)
print('Rules before:', len(rules), '| after:', len(nonredundant_rules))
display(nonredundant_rules[['antecedents','consequents','support','confidence','lift']].head(15) if not nonredundant_rules.empty else nonredundant_rules)

In [ ]:
# Custom: Hasse edges of the frequent-itemset inclusion lattice.
# Immediate inclusion is obtained by removing exactly one item from each itemset.
def lattice_edges(itemsets):
    nodes = set(itemsets['itemsets'])
    edges = []
    for upper in nodes:
        if len(upper) < 2:
            continue
        for item in upper:
            lower = upper - {item}
            if lower in nodes:
                edges.append((lower, upper))
    return pd.DataFrame(edges, columns=['subset', 'superset'])

lattice = lattice_edges(frequent_itemsets)
print('Lattice edges:', len(lattice))
display(lattice.head(10))

3.4 Objective
Interest Evaluation

In [ ]:
singleton_support = basket_df.mean(axis=0).to_dict()


def itemset_interest_metrics(itemsets, singleton_support, h=0.6, cross_threshold=0.3):
    out = itemsets.copy()

    # Convert integer column indices to item names
    out['all_confidence'] = out.apply(
        lambda r: r['support'] / max(
            singleton_support[basket_df.columns[i]]
            for i in r['itemsets']
        ),
        axis=1
    )

    # Cross-support ratio
    out['cross_support_ratio'] = out['itemsets'].apply(
        lambda x: min(
            singleton_support[basket_df.columns[i]]
            for i in x
        ) / max(
            singleton_support[basket_df.columns[i]]
            for i in x
        )
    )

    # Identify hypercliques
    out['hyperclique'] = out['all_confidence'] >= h

    # Identify cross-support itemsets
    out['cross_support'] = (
        (out['itemsets'].apply(len) >= 2)
        & (out['cross_support_ratio'] < cross_threshold)
    )

    return out


itemset_metrics = itemset_interest_metrics(
    frequent_itemsets,
    singleton_support,
    H_THRESHOLD,
    CROSS_SUPPORT_THRESHOLD
)

hypercliques = itemset_metrics.loc[
    (itemset_metrics['hyperclique'])
    & (itemset_metrics['itemsets'].apply(len) >= 2)
].copy()

cross_support_itemsets = itemset_metrics.loc[
    itemset_metrics['cross_support']
].copy()

print("Hypercliques:", len(hypercliques))
print("Cross-support itemsets:", len(cross_support_itemsets))

display(hypercliques.head())
display(cross_support_itemsets.head())


In [ ]:
def add_rule_interest_metrics(rules, itemset_metrics):
    out = rules.copy()
    if out.empty:
        return out
    
    out['cosine'] = out['support'] / np.sqrt(out['antecedent support'] * out['consequent support'])
    
    out['interest'] = out['support'] - out['antecedent support'] * out['consequent support']
    
    ac = dict(zip(itemset_metrics['itemsets'], itemset_metrics['all_confidence']))
    
    csr = dict(zip(itemset_metrics['itemsets'], itemset_metrics['cross_support_ratio']))
    
    out['all_confidence'] = out.apply(lambda r: ac[r['antecedents'] | r['consequents']], axis=1)
    out['cross_support_ratio'] = out.apply(lambda r: csr[r['antecedents'] | r['consequents']], axis=1)
    out['hyperclique'] = out['all_confidence'] >= H_THRESHOLD
    out['cross_support'] = out['cross_support_ratio'] < CROSS_SUPPORT_THRESHOLD
    
    return out

rules_scored = add_rule_interest_metrics(rules, itemset_metrics)

nonredundant_scored = add_rule_interest_metrics(nonredundant_rules, itemset_metrics)

columns = ['antecedents','consequents','support','confidence','lift','cosine','interest','all_confidence','cross_support_ratio','hyperclique','cross_support']

display(rules_scored[columns].head(20) if not rules_scored.empty else rules_scored)

Part II: AI Integration Protocol and Prompt Logging

prompt: generate code for candidate rule generation using Apriori

    the AI skipped rule generation no support counting, rules or confidence

In [ ]:
from itertools import combinations

def apriori_gen(L_prev):
    candidates = set()

    # Change 1: did nothing, because sorted() compares frozensets by subset, not by their items
    L_list = sorted(L_prev, key=sorted)

    for i in range(len(L_list)):
        l1 = sorted(L_list[i])
        for j in range(i + 1, len(L_list)):
            l2 = sorted(L_list[j])

            # Change 2: once the first k-2 items stop matching, no later itemset can match
            if l1[:-1] != l2[:-1]:
                break

            candidate = L_list[i] | L_list[j]

            # Change 3: check subsets against the original set L_prev, instant lookup vs item by item
            if all(frozenset(s) in L_prev for s in combinations(candidate, len(candidate) - 1)):
                candidates.add(candidate)

    return candidates

L2 = {frozenset(p) for p in [("A","B"), ("A","C"), ("A","D"), ("B","C"), ("B","D"), ("C","D")]}
for c in sorted(apriori_gen(L2), key=sorted):
       print(sorted(c))